In [ ]:
# Code2Video Colab Runner
#
# This notebook runs /content/Code2Video with the same OpenRouter GPT-5.5 API setup
# and the first three topics from topics.xlsx.
#
# Expected Google Drive inputs:
# - /content/drive/MyDrive/education_video/Code2Video.zip
# - /content/drive/MyDrive/education_video/topics.xlsx
#
# Persistent output destination:
# - /content/drive/MyDrive/education_video/c2v_output

In [ ]:
# 1) Install system dependencies for Manim rendering.
# This can take several minutes on a fresh Colab runtime.

!apt-get update -qq
!apt-get install -y -qq ffmpeg libcairo2-dev libpango1.0-dev pkg-config python3-dev texlive-latex-base texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended dvisvgm

print("System dependencies installed.")

In [ ]:
# 2) Prepare Code2Video and topics.xlsx from Google Drive.
# Put these files in Drive first:
#   MyDrive/education_video/Code2Video.zip
#   MyDrive/education_video/topics.xlsx

from pathlib import Path
import os
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

content_dir = Path("/content")
drive_input_dir = Path("/content/drive/MyDrive/education_video")
project_zip = drive_input_dir / "Code2Video.zip"
drive_topics = drive_input_dir / "topics.xlsx"
project_dir = content_dir / "Code2Video"
topics_xlsx = content_dir / "topics.xlsx"

if not project_zip.exists():
    raise FileNotFoundError(f"Missing project zip: {project_zip}")
if not drive_topics.exists():
    raise FileNotFoundError(f"Missing topics spreadsheet: {drive_topics}")

if project_dir.exists():
    shutil.rmtree(project_dir)
with zipfile.ZipFile(project_zip, "r") as zf:
    zf.extractall(content_dir)

# Handle zips that contain either Code2Video/ or a single top-level folder.
if not project_dir.exists():
    candidates = [p for p in content_dir.iterdir() if p.is_dir() and p.name.startswith("Code2Video")]
    if candidates:
        candidates[0].rename(project_dir)

shutil.copy2(drive_topics, topics_xlsx)

if not (project_dir / "src" / "agent.py").exists():
    raise FileNotFoundError(f"Code2Video source not found under: {project_dir}")

print("Project:", project_dir)
print("Topics:", topics_xlsx)

In [ ]:
# 3) Install Python dependencies.

from pathlib import Path

project_dir = Path("/content/Code2Video")
requirements = project_dir / "src" / "requirements.txt"
if not requirements.exists():
    raise FileNotFoundError(requirements)

%cd /content/Code2Video/src
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q openpyxl

print("Python dependencies installed.")

In [ ]:
# 4) Runtime configuration.

from pathlib import Path
from datetime import datetime
import os

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from colab_bootstrap import ensure_openai_key, setup_api_tracking, openai_model, print_api_usage

ensure_openai_key()
setup_api_tracking()
OPENAI_MODEL = openai_model()

PROJECT_ROOT = Path("/content/Code2Video")
PROJECT_SRC = PROJECT_ROOT / "src"
TOPICS_XLSX = Path("/content/topics.xlsx")

MAX_TOPICS = 0  # 0 = all remaining topics
MAX_CONCURRENT_LLM = int(os.getenv("BASELINE_MAX_CONCURRENT_LLM", "20"))
FOLDER_PREFIX = "c2v_openai_gpt4o"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOCAL_OUTPUT_ROOT = Path("/content/c2v_output") / RUN_ID
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/education_video/c2v_output")

# Keep these off by default so the run depends only on the text OpenRouter model.
# Code2Video still uses its original Planner/Coder/Manim flow.
USE_FEEDBACK = False
USE_ASSETS = False

os.environ["OPENAI_MODEL"] = OPENAI_MODEL

LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Model:", OPENAI_MODEL)
print("Project root:", PROJECT_ROOT)
print("Project src:", PROJECT_SRC)
print("Topics:", TOPICS_XLSX)
print("Max topics:", MAX_TOPICS)
print("Max concurrent LLM:", MAX_CONCURRENT_LLM)
print("Local output:", LOCAL_OUTPUT_ROOT)
print("Drive output:", DRIVE_OUTPUT_ROOT)

In [ ]:
# 5) OpenRouter smoke test.

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: ok"}],
    max_tokens=32,
)
print(resp.choices[0].message.content)
print("OpenRouter smoke test passed.")

In [ ]:
# 6) Write Code2Video OpenRouter runner.
# It reads the first MAX_TOPICS valid rows from topics.xlsx and runs Code2Video with LLM concurrency set to MAX_CONCURRENT_LLM.

from pathlib import Path

runner_path = PROJECT_SRC / "c2v_openrouter_runner.py"
runner_path.write_text(r'''
from __future__ import annotations

import argparse
import csv
import json
import os
import shutil
import sys
import time
from pathlib import Path
from typing import Any

from openai import OpenAI
from openpyxl import load_workbook


def clean(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def load_topic_rows(xlsx_path: Path, max_topics: int) -> list[dict[str, str]]:
    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    rows = ws.iter_rows(values_only=True)
    header = [clean(v) for v in next(rows)]

    out: list[dict[str, str]] = []
    for row_idx, raw_row in enumerate(rows, start=2):
        item = {header[i]: clean(raw_row[i]) if i < len(raw_row) else "" for i in range(len(header))}
        request_id = item.get("Request ID", "")
        topic = item.get("Topic", "")
        persona = item.get("Student Persona", "")
        if not request_id or not topic or not persona:
            continue
        if request_id == "Request ID" or topic == "Topic":
            continue
        item["_row_number"] = str(row_idx)
        out.append(item)
        if max_topics is not None and max_topics > 0 and len(out) >= max_topics:
            break
    return out


def request_openai_token(prompt, log_id=None, max_tokens=10000, max_retries=3):
    api_key = os.environ["OPENAI_API_KEY"]
    model = os.environ.get("OPENAI_MODEL", "gpt-4o")
    client = OpenAI(api_key=api_key)

    usage_info = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max(32, int(max_tokens or 10000)),
                extra_headers={},
            )
            if completion.usage:
                usage_info["prompt_tokens"] = getattr(completion.usage, "prompt_tokens", 0) or 0
                usage_info["completion_tokens"] = getattr(completion.usage, "completion_tokens", 0) or 0
                usage_info["total_tokens"] = getattr(completion.usage, "total_tokens", 0) or 0
            return completion, usage_info
        except Exception as exc:
            last_error = exc
            print(f"OpenAI request failed on attempt {attempt}/{max_retries}: {exc}", flush=True)
            if attempt < max_retries:
                time.sleep(min(8, 2 ** attempt))
    raise RuntimeError(f"OpenAI request failed after {max_retries} attempts: {last_error}")


def run(args: argparse.Namespace) -> None:
    project_root = Path(args.project_root).resolve()
    project_src = Path(args.project_src).resolve()
    topics_xlsx = Path(args.topics_xlsx).resolve()
    local_output_root = Path(args.local_output_root).resolve()
    drive_output_root = Path(args.drive_output_root).resolve()

    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(project_src))
    os.chdir(project_root)

    import agent as agent_module
    agent_module.api = request_openai_token
    from agent import RunConfig, run_Code2Video

    rows = load_topic_rows(topics_xlsx, args.max_topics)
    sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
    from common_topics import topic_has_video
    pending_rows = []
    for row in rows:
        req_id = row["Request ID"]
        if topic_has_video(drive_output_root, req_id):
            print(f"SKIP (done): [{req_id}] {row['Topic']}", flush=True)
            continue
        pending_rows.append(row)
    rows = pending_rows
    knowledge_points = [row["Topic"] for row in rows]

    local_output_root.mkdir(parents=True, exist_ok=True)
    drive_output_root.mkdir(parents=True, exist_ok=True)
    metadata_path = local_output_root / "input_rows.json"
    metadata_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

    # Code2Video derives json_files/assets from the path prefix before CASES,
    # so CASES must live under the project root, not under src/.
    cases_folder = project_root / "CASES" / args.folder_prefix
    if cases_folder.exists() and args.force:
        shutil.rmtree(cases_folder)
    cases_folder.mkdir(parents=True, exist_ok=True)

    cfg = RunConfig(
        api=request_openai_token,
        iconfinder_api_key="",
        use_feedback=args.use_feedback,
        use_assets=args.use_assets,
        max_code_token_length=args.max_code_token_length,
        max_fix_bug_tries=args.max_fix_bug_tries,
        max_regenerate_tries=args.max_regenerate_tries,
        max_feedback_gen_code_tries=args.max_feedback_gen_code_tries,
        max_mllm_fix_bugs_tries=args.max_mllm_fix_bugs_tries,
        feedback_rounds=args.feedback_rounds,
    )

    manifest_path = local_output_root / "manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["request_id", "topic", "row_number"])
        writer.writeheader()
        for row in rows:
            writer.writerow({"request_id": row["Request ID"], "topic": row["Topic"], "row_number": row["_row_number"]})

    print(f"Running {len(knowledge_points)} topics with max concurrency {args.max_concurrent_llm}", flush=True)
    print("Topics:", json.dumps(knowledge_points, ensure_ascii=False, indent=2), flush=True)

    run_Code2Video(
        knowledge_points,
        cases_folder,
        parallel=True,
        batch_size=1,
        max_workers=args.max_concurrent_llm,
        cfg=cfg,
    )

    copied_cases = local_output_root / "CASES" / args.folder_prefix
    copied_cases.parent.mkdir(parents=True, exist_ok=True)
    if copied_cases.exists():
        shutil.rmtree(copied_cases)
    shutil.copytree(cases_folder, copied_cases)

    zip_path = shutil.make_archive(str(local_output_root), "zip", root_dir=local_output_root)

    drive_run_dir = drive_output_root / local_output_root.name
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    shutil.copytree(local_output_root, drive_run_dir)
    drive_zip = drive_output_root / Path(zip_path).name
    shutil.copy2(zip_path, drive_zip)

    print("Local output:", local_output_root, flush=True)
    print("Drive output folder:", drive_run_dir, flush=True)
    print("Drive zip:", drive_zip, flush=True)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--project-root", required=True)
    parser.add_argument("--project-src", required=True)
    parser.add_argument("--topics-xlsx", required=True)
    parser.add_argument("--local-output-root", required=True)
    parser.add_argument("--drive-output-root", required=True)
    parser.add_argument("--folder-prefix", required=True)
    parser.add_argument("--max-topics", type=int, default=0)
    parser.add_argument("--max-concurrent-llm", type=int, default=int(os.getenv("BASELINE_MAX_CONCURRENT_LLM", "20")))
    parser.add_argument("--max-code-token-length", type=int, default=10000)
    parser.add_argument("--max-fix-bug-tries", type=int, default=10)
    parser.add_argument("--max-regenerate-tries", type=int, default=10)
    parser.add_argument("--max-feedback-gen-code-tries", type=int, default=3)
    parser.add_argument("--max-mllm-fix-bugs-tries", type=int, default=3)
    parser.add_argument("--feedback-rounds", type=int, default=2)
    parser.add_argument("--use-feedback", action="store_true")
    parser.add_argument("--use-assets", action="store_true")
    parser.add_argument("--force", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    run(parse_args())
''', encoding="utf-8")

print("Wrote runner:", runner_path)

In [ ]:
# 7) Run Code2Video for the first three topics.

import os
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    str(PROJECT_SRC / "c2v_openrouter_runner.py"),
    "--project-root", str(PROJECT_ROOT),
    "--project-src", str(PROJECT_SRC),
    "--topics-xlsx", str(TOPICS_XLSX),
    "--local-output-root", str(LOCAL_OUTPUT_ROOT),
    "--drive-output-root", str(DRIVE_OUTPUT_ROOT),
    "--folder-prefix", FOLDER_PREFIX,
    "--max-topics", str(MAX_TOPICS),
    "--max-concurrent-llm", str(MAX_CONCURRENT_LLM),
    "--force",
]

if USE_FEEDBACK:
    cmd.append("--use-feedback")
if USE_ASSETS:
    cmd.append("--use-assets")

print("Running:", " ".join(cmd), flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    rc = process.wait()
    if rc != 0:
        print(f"Runner failed with exit code {rc}. Scroll up in this cell for the real error above.", flush=True)
        raise subprocess.CalledProcessError(rc, cmd)
finally:
    if process.poll() is None:
        process.terminate()

print("Done. Local output:", LOCAL_OUTPUT_ROOT)
print("Done. Drive output:", DRIVE_OUTPUT_ROOT)

In [ ]:
# 8) Copy latest c2v_output to the education_video folder in Google Drive.
# Useful after interruption, kernel restart, or older runs that wrote to MyDrive/c2v_output.

from pathlib import Path
import shutil

local_base = Path("/content/c2v_output")
old_drive_root = Path("/content/drive/MyDrive/c2v_output")
drive_root = Path("/content/drive/MyDrive/education_video/c2v_output")
drive_root.mkdir(parents=True, exist_ok=True)

src_dir = None
if "LOCAL_OUTPUT_ROOT" in globals() and Path(LOCAL_OUTPUT_ROOT).exists():
    src_dir = Path(LOCAL_OUTPUT_ROOT)
else:
    candidates = []
    if local_base.exists():
        candidates.extend([p for p in local_base.iterdir() if p.is_dir()])
    if old_drive_root.exists():
        candidates.extend([p for p in old_drive_root.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError("No c2v output found under /content/c2v_output or /content/drive/MyDrive/c2v_output")
    src_dir = max(candidates, key=lambda p: p.stat().st_mtime)

dst_dir = drive_root / src_dir.name
if dst_dir.exists():
    shutil.rmtree(dst_dir)
shutil.copytree(src_dir, dst_dir)

zip_candidates = [src_dir.with_suffix(".zip")]
if old_drive_root.exists():
    zip_candidates.append(old_drive_root / f"{src_dir.name}.zip")
zip_path = next((p for p in zip_candidates if p.exists()), None)
if zip_path is None:
    zip_path = Path(shutil.make_archive(str(src_dir), "zip", root_dir=src_dir))

dst_zip = drive_root / zip_path.name
shutil.copy2(zip_path, dst_zip)

print("Copied folder to:", dst_dir)
print("Copied zip to:", dst_zip)